## MT 1D FORWARD MODELING PROGRAM

In [9]:
import math
import cmath
import numpy as np

In [10]:
mu = 4*math.pi*1E-7; #Magnetic Permeability (H/m)
print("This is the magnetic permeability:" + str(mu))

This is the magnetic permeability:1.2566370614359173e-06


Now create a 5 layers model, whose resistivities are, from top to bottom: 300, 2500, 0.8, 3000 and 2500 Ohm.m, and whose thicknesses are: 200, 400, 40 and 500 m.  FYI, the program will accept any number of layers, but you must ensure that if N is the number of resistivities, the thicknesses array must have N-1 elements

In [11]:
resistivities=[300, 2500, 0.8, 3000, 2500]
thicknesses=[200, 400, 40, 500]
n=len(resistivities)

Let's write a function to compute the impedance for each layer, for a given frequency:

In [12]:
def compute_apparent_resistivity_and_phase(frequency):
    w =  2*math.pi*frequency;       
    impedances = list(range(n));
    #compute basement impedance
    impedances[n-1] = cmath.sqrt(w*mu*resistivities[n-1]*1j);

    for j in range(n-2,-1,-1):
        resistivity = resistivities[j];
        thickness = thicknesses[j];
  
        # 3. Compute apparent resistivity from top layer impedance
        #Step 2. Iterate from bottom layer to top(not the basement) 
        # Step 2.1 Calculate the intrinsic impedance of current layer
        dj = cmath.sqrt((w * mu * (1.0/resistivity))*1j);
        wj = dj * resistivity;
        # Step 2.2 Calculate Exponential factor from intrinsic impedance
        ej = cmath.exp(-2*thickness*dj);                     
    
        # Step 2.3 Calculate reflection coeficient using current layer
        #          intrinsic impedance and the below layer impedance
        belowImpedance = impedances[j + 1];
        rj = (wj - belowImpedance)/(wj + belowImpedance);
        re = rj*ej; 
        Zj = wj * ((1 - re)/(1 + re));
        impedances[j] = Zj;
        ### Compute apparent resistivity from top layer impedance
    Z = impedances[0];
    absZ = abs(Z);
    apparentResistivity = (absZ * absZ)/(mu * w);
    phase = math.atan2(Z.imag, Z.real)
    print(frequency, '\t', apparentResistivity, '\t', phase);
    return frequency, apparentResistivity, phase
    

In [13]:
frequencies = [0.0001,0.005,0.01,0.05,0.1,0.5,1,5,10,50,100,500,10000];

In [14]:
apparentResistivities=np.zeros_like(frequencies);
phases=np.zeros_like(frequencies);

In [15]:
for i, frequency in enumerate(frequencies): 
  print(i)
  w =  2*math.pi*frequency;       
  impedances = list(range(n));
  frequency, apparentResistivity, phase = compute_apparent_resistivity_and_phase(frequency)
  apparentResistivities[i]=apparentResistivity
  phases[i]=phase


0
0.0001 	 2261.517499567936 	 0.7376757083961615
1
0.005 	 1274.7285512555263 	 0.5312155926906661
2
0.01 	 997.7172890535873 	 0.46670795682516664
3
0.05 	 435.8126507144753 	 0.31540795482698775
4
0.1 	 273.8525552239266 	 0.2658315915298133
5
0.5 	 78.49519062166195 	 0.2566784856953265
6
1 	 45.23031947523922 	 0.34266077819292307
7
5 	 24.458612784461803 	 0.9068714331972648
8
10 	 34.234784851703914 	 1.1702828201355762
9
50 	 143.13776091917669 	 1.3703073536080494
10
100 	 270.13876061773254 	 1.2965108376371062
11
500 	 533.3425959747472 	 0.7788825973447078
12
10000 	 299.2377172466316 	 0.7958034422163801


Now modify the function compute_apparent_resistivity_and_phase function to return the resistivity and phase values and plot those.

What have you modelled here: the apparent or the true resistivity?
How does this quantity vary with frequency? Does it make sense?

This tutorial was freely adapted from Andrew Pethik's tutorial.